# Анализ A/B-теста изменения стоимости Premium-подписки

## 1. Постановка задачи

### Бизнес-контекст

Компания разрабатывает мобильное приложение для знакомств. Одним из источников монетизации является Premium-подписка, предоставляющая пользователям дополнительные возможности.

Для новых пользователей из ряда стран была изменена стоимость Premium-подписки при покупке через две новые платёжные системы. Стоимость пробного периода осталась прежней.

### Цель исследования

Оценить влияние изменения стоимости Premium-подписки на поведение пользователей и выручку приложения.

### Основные вопросы исследования

1. Как изменилась конверсия в покупку Premium?
2. Повлияло ли изменение цены на ARPU и ARPPU?
3. Являются ли наблюдаемые различия статистически значимыми?
4. Насколько устойчивы результаты к особенностям данных?
5. Можно ли рекомендовать внедрение новой цены?

### Подход к анализу

Для исследования используются данные трёх экспериментальных групп: Test, Control 1 и Control 2.

Анализ включает проверку качества данных, расчёт продуктовых метрик, статистическое тестирование, визуализацию результатов и формирование бизнес-рекомендаций.

## 2. Описание исходных данных

Для исследования предоставлены шесть CSV-файлов, содержащих информацию о пользователях и транзакциях трёх экспериментальных групп.

### 2.1. Экспериментальные группы

- **Test** — пользователи тестовой группы, для которых проводился эксперимент с изменением стоимости Premium-подписки.
- **Control 1** — первая контрольная группа.
- **Control 2** — вторая контрольная группа.

Две контрольные группы позволяют дополнительно проверить сопоставимость результатов в условиях без изменения цены.

### 2.2. Данные о пользователях

Три файла: users_test, users_control_1 и users_control_2.

Основные поля:

- `uid` — идентификатор пользователя.
- `age` — возраст.
- `gender` — пол.
- `country` — страна.
- `attraction_coeff` — коэффициент привлекательности.
- `visit_days` — дни посещения приложения.
- `was_premium` — информация о наличии Premium в прошлом.
- `is_premium` — текущий статус Premium.
- `total_revenue` — общая нормированная выручка пользователя.

Также присутствуют дополнительные характеристики, связанные с активностью пользователей и настройками поиска.

### 2.3. Данные о транзакциях

Три файла: transactions_test, transactions_control_1 и transactions_control_2.

Основные поля:

- `uid` — идентификатор пользователя.
- `country` — страна.
- `joined_at` — дата регистрации пользователя.
- `paid_at` — дата платежа.
- `revenue` — нормированная сумма платежа.
- `payment_id` — идентификатор платёжной системы.
- `from_page` — источник совершения платежа.
- `product_type` — тип приобретённого продукта.

### 2.4. Особенности данных

В таблицах пользователей каждая строка соответствует отдельному пользователю.

В таблицах транзакций один пользователь может встречаться несколько раз, поскольку способен совершить несколько покупок.

Для расчёта пользовательских метрик необходимо агрегировать транзакции по `uid`, сохраняя пользователей без покупок с нулевой выручкой.

Перед анализом требуется проверить пропуски, дубликаты, корректность дат и согласованность выручки между источниками.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import confint_proportions_2indep
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
from IPython.display import display

In [ ]:
# Загрузка исходных данных
DATA_DIR = Path('data/private')
expected_files = [f'{kind}_{group}.csv' for kind in ['users', 'transactions']
                  for group in ['test', 'control_1', 'control_2']]
missing = [name for name in expected_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Отсутствуют приватные файлы в {DATA_DIR.resolve()}: {missing}. '
        'См. data/README.md; запускайте ноутбук из корня проекта.'
    )

users_test = pd.read_csv(
    DATA_DIR / 'users_test.csv', sep=';'
)
users_control_1 = pd.read_csv(
    DATA_DIR / 'users_control_1.csv', sep=';'
)
users_control_2 = pd.read_csv(
    DATA_DIR / 'users_control_2.csv', sep=';'
)

transactions_test = pd.read_csv(
    DATA_DIR / 'transactions_test.csv', sep=';'
)
transactions_control_1 = pd.read_csv(
    DATA_DIR / 'transactions_control_1.csv', sep=';'
)
transactions_control_2 = pd.read_csv(
    DATA_DIR / 'transactions_control_2.csv', sep=';'
)


In [ ]:
users = {
    'Test': users_test,
    'Control 1': users_control_1,
    'Control 2': users_control_2}

transactions = {
    'Test': transactions_test,
    'Control 1': transactions_control_1,
    'Control 2': transactions_control_2}

In [ ]:
# Размеры исходных пользовательских таблиц
for name, df in users.items():
    print(f'{name}: {df.shape[0]} пользователей, {df.shape[1]} столбцов')

print()

# Размеры исходных таблиц транзакций
for name, df in transactions.items():
    print(f'{name}: {df.shape[0]} строк транзакций, {df.shape[1]} столбцов')

In [ ]:
users_test.head()

In [ ]:
users_test.info()

## 3. Проверка и предобработка данных

Перед проведением A/B-теста необходимо проверить качество исходных данных. Ошибки и некорректные записи могут повлиять на расчёт метрик и достоверность результатов исследования.

### 3.1. Анализ пропущенных значений

На первом этапе проверим наличие пропусков в пользовательских данных и определим, какие из них могут быть связаны с особенностями хранения информации, а какие требуют дополнительной обработки.

In [ ]:
users_test.isna().sum()

In [ ]:
users_test['is_premium'].value_counts(dropna=False)

In [ ]:
users_test['was_premium'].value_counts(dropna=False)

In [ ]:
pd.crosstab(
    users_test['was_premium'].fillna(-1),
    users_test['is_premium'].fillna(-1)
)

**Результаты анализа пропусков**

В тестовой группе обнаружены пропущенные значения в нескольких столбцах:

- `coins` — 4 259 пропусков.
- `visit_days` — 1 330 пропусков.
- `was_premium` — 3 900 пропусков.
- `is_premium` — 4 151 пропуск.

Дополнительно исследованы признаки `was_premium` и `is_premium`. В обоих столбцах встречаются значения `1` и `NaN`.

В тестовой группе 157 пользователей имеют отметку о текущем Premium, а 251 пользователь ранее имел Premium, но не имеет отметки о действующей подписке.

Вероятно, пропуски в этих признаках обозначают отсутствие соответствующего статуса. Однако без подтверждения документацией данных это остаётся предположением.

**Вывод:** обнаруженные пропуски не удалялись автоматически, поскольку часть из них может отражать особенности хранения пользовательских данных.

In [ ]:
users_control_1['is_premium'].value_counts(dropna=False)

In [ ]:
users_control_2['is_premium'].value_counts(dropna=False)

In [ ]:
premium_rate = users_test['is_premium'].eq(1).mean() * 100
print(f'Доля Premium в тестовой группе: {premium_rate:.2f}%')

### 3.2. Проверка уникальности пользователей

Проверим наличие повторяющихся идентификаторов пользователей внутри каждой экспериментальной группы.

Дополнительно проверим пересечения между группами, поскольку один пользователь не должен одновременно участвовать в нескольких вариантах эксперимента.


In [ ]:
users_test['uid'].duplicated().sum()

In [ ]:
users_control_1['uid'].duplicated().sum()

In [ ]:
users_control_2['uid'].duplicated().sum()

In [ ]:
users_test['uid'].isin(users_control_1['uid']).sum()

In [ ]:
users_test['uid'].isin(users_control_2['uid']).sum()

In [ ]:
users_control_1['uid'].isin(users_control_2['uid']).sum()

**Результаты проверки**

- Дубликатов `uid` внутри экспериментальных групп не обнаружено.
- Пересечений пользователей между Test, Control 1 и Control 2 не выявлено.

**Вывод:** каждый пользователь представлен в своей группе один раз и не встречается в других группах. По этому критерию разделение пользователей выполнено корректно.

При этом отсутствие дубликатов и пересечений само по себе не гарантирует полной сопоставимости экспериментальных групп.


### 3.3. Проверка и очистка транзакций

#### 3.3.1. Анализ пропущенных значений

Проверим таблицы транзакций на наличие пропусков и некорректных записей.

Особое внимание уделим полям `uid` и `revenue`, поскольку они необходимы для сопоставления платежей с пользователями и расчёта денежных метрик.


In [ ]:
for name, df in transactions.items():
    print(f'\n{name}: {df.shape[0]} строк')
    print(df.isna().sum())

In [ ]:
transactions_control_1[
    transactions_control_1['uid'].isna()].head()

In [ ]:
# Удаляем строки без uid и revenue
transactions_control_1_clean = transactions_control_1.dropna(
    subset=['uid', 'revenue']
).copy()

# Проверяем результат
print('До очистки:', len(transactions_control_1))
print('После очистки:', len(transactions_control_1_clean))

**Результаты проверки**

В таблице `transactions_control_1` обнаружено 630 строк без идентификатора пользователя и суммы платежа.

Эти записи не содержат информации, необходимой для анализа выручки, поэтому были исключены из рабочей таблицы.

После очистки количество строк сократилось с 1007 до 377.

В таблицах Test и Control 2 аналогичных пропусков не обнаружено.

**Вывод:** исключены 630 некорректных записей. Исходные данные сохранены без изменений.


#### 3.3.2. Анализ дубликатов транзакций

Проверим наличие полностью повторяющихся записей в таблицах транзакций.

Отдельно исследуем поле `payment_id`, чтобы определить, можно ли использовать его в качестве уникального идентификатора отдельного платежа.

Удаление повторов будем рассматривать только после проверки их влияния на выручку.


In [ ]:
# Проверяем полные дубликаты транзакций
for name, df in transactions.items():
    duplicates = df.duplicated().sum()
    print(f'{name}: {duplicates} дубликатов')

In [ ]:
# Все экземпляры повторяющихся строк
transactions_test[
    transactions_test.duplicated(keep=False)].sort_values('uid').head(10)

In [ ]:
# Исследуем идентификаторы платежей
for name, df in transactions.items():
    print(f'\n{name}')
    print('Всего транзакций:', len(df))
    print('Уникальных payment_id:', df['payment_id'].nunique())
    print(df['payment_id'].value_counts().head(3))

In [ ]:
# Создаём словарь для очищенных таблиц
transactions_clean = {}

for name, df in transactions.items():
    # Удаляем полные дубликаты
    transactions_clean[name] = df.drop_duplicates().copy()

    # Сравниваем количество строк
    print(f'{name}:')
    print(f'До очистки: {len(df)}')
    print(f'После очистки: {len(transactions_clean[name])}')
    print()

In [ ]:
# Сравниваем выручку из двух источников
for name in users:
    users_revenue = users[name]['total_revenue'].sum()
    transactions_revenue = transactions_clean[name]['revenue'].sum()

    print(f'{name}:')
    print(f'Выручка пользователей: {users_revenue}')
    print(f'Выручка транзакций: {transactions_revenue}')
    print(f'Разница: {users_revenue - transactions_revenue}')
    print()

In [ ]:
# Сравниваем выручку до и после удаления дубликатов
for name in transactions:
    revenue_before = transactions[name]['revenue'].sum()
    revenue_after = transactions_clean[name]['revenue'].sum()

    print(f'{name}:')
    print(f'До удаления: {revenue_before:.0f}')
    print(f'После удаления: {revenue_after:.0f}')
    print(f'Разница: {revenue_before - revenue_after:.0f}')
    print()

**Результаты проверки**

Обнаружены полные дубликаты транзакций:

- Test — 7 повторных строк.
- Control 1 — 21 повторная строка.
- Control 2 — 5 повторных строк.

Всего обнаружено 33 повторных вхождения в исходных таблицах, включая записи с пропусками. В расчётах после удаления 630 некорректных строк используется очищенная Control 1.

Проверка показала, что `payment_id` повторяется у разных транзакций и не позволяет однозначно идентифицировать отдельный платёж.

После удаления полных дубликатов возникло расхождение между суммой транзакций и `total_revenue` в Control 2. При сохранении повторов выручка совпадает даже на уровне отдельных пользователей.

**Вывод:** полные повторы не были автоматически исключены из основного анализа, поскольку нет достаточных оснований считать все такие записи техническими ошибками.

Для оценки устойчивости результатов необходимо дополнительно сравнить метрики с учётом повторов и без них.


In [ ]:
# Суммируем платежи каждого пользователя
payments_by_user = transactions_control_2.groupby(
    'uid', as_index=False
)['revenue'].sum()

# Объединяем таблицы по uid
comparison = users_control_2[
    ['uid', 'total_revenue']
].merge(
    payments_by_user,
    on='uid',
    how='left'
)

# Пользователям без платежей ставим выручку 0
comparison['revenue'] = comparison['revenue'].fillna(0)

# Рассчитываем разницу
comparison['difference'] = (
    comparison['total_revenue'] - comparison['revenue']
)

# Считаем пользователей с расхождениями
print('Пользователей с расхождениями:',
      (comparison['difference'] != 0).sum())

# Показываем первые 10 расхождений
comparison[comparison['difference'] != 0].head(10)

In [ ]:
# Проверяем, все ли транзакции принадлежат участникам своих групп

for name in users:
    unknown = transactions[name][
        ~transactions[name]['uid'].isin(users[name]['uid'])
    ]

    print(f'{name}:')
    print(f'Чужих транзакций: {len(unknown)}')
    print(f'Их выручка: {unknown["revenue"].sum():.0f}')
    print()
# Используем для последующих метрик только строки с uid и revenue.
# Исходный transactions_control_1 остаётся неизменённым для аудита.
transactions = {
    'Test': transactions_test,
    'Control 1': transactions_control_1_clean,
    'Control 2': transactions_control_2
}

### 3.4. Проверка согласованности выручки

В данных присутствуют два источника информации о выручке: `total_revenue` в таблицах пользователей и `revenue` в таблицах транзакций.

Для проверки согласованности данных сравним суммарную выручку по каждой экспериментальной группе, а затем проведём сверку на уровне отдельных пользователей.

Это позволит обнаружить расхождения, которые могут повлиять на расчёт ARPU и интерпретацию результатов A/B-теста.

In [ ]:
# Сравниваем выручку пользователей в Test и Control 1

for name in ['Test', 'Control 1']:

    # Суммируем транзакции каждого пользователя
    payments = transactions[name].groupby(
        'uid', as_index=False
    )['revenue'].sum()

    # Соединяем с таблицей пользователей
    comparison = users[name][
        ['uid', 'total_revenue']].merge(payments, on='uid', how='left')

    # Заполняем отсутствие платежей нулями
    comparison['revenue'] = comparison['revenue'].fillna(0)

    # Вычисляем разницу
    comparison['difference'] = (
        comparison['total_revenue'] - comparison['revenue']
    )

    # Оставляем только несовпадения
    mismatches = comparison[comparison['difference'] != 0]

    print(f'{name}: {len(mismatches)} пользователей с расхождениями')
    display(mismatches)

**Результаты сверки выручки**

При сравнении агрегированной выручки пользователей с суммой исходных транзакций обнаружены расхождения у трёх пользователей:

- **Test:** один пользователь. Разница составляет 44 083 единицы. Обнаружена транзакция с датой, предшествующей регистрации пользователя.
- **Control 1:** два пользователя. Расхождения составляют 780 и 30 264 единицы. Они связаны с операциями, которые требуют дополнительной проверки.
- **Control 2:** расхождений не обнаружено. Выручка совпадает на уровне каждого пользователя при сохранении повторяющихся транзакций.

Таким образом, основная часть данных о выручке согласована между двумя источниками.

**Вывод:** обнаруженные расхождения не позволяют однозначно определить ошибочные операции. Поэтому в основном анализе для расчёта общего ARPU используется `total_revenue`, а для расчёта ARPU Premium — информация о соответствующих транзакциях.

Различия между источниками и возможное влияние спорных платежей учитываются как ограничения исследования.

In [ ]:
# Пользователи с расхождениями
problem_users = {
    'Test': [892275462],
    'Control 1': [960936960, 892355163]
}

for name, uids in problem_users.items():
    print(f'\n{name}')

    # Выбираем транзакции проблемных пользователей
    problem_transactions = transactions[name][
        transactions[name]['uid'].isin(uids)
    ]

    # Сортируем платежи по дате
    display(
        problem_transactions.sort_values(['uid', 'paid_at']))

### 3.5. Проверка корректности дат платежей

Проверим хронологическую согласованность транзакций. В частности, выясним, встречаются ли платежи, совершённые раньше даты регистрации пользователя.

Такие записи могут указывать на ошибки в исходных данных или некорректную интерпретацию формата дат.

Перед исключением подозрительных транзакций необходимо исследовать возможные причины расхождений.


In [ ]:
# Проверяем платежи до даты регистрации

for name, df in transactions.items():

    # Создаём копию таблицы
    data = df.copy()

    # Преобразуем текстовые даты в datetime
    data['joined_at'] = pd.to_datetime(data['joined_at'])
    data['paid_at'] = pd.to_datetime(data['paid_at'])

    # Находим платежи раньше регистрации
    invalid_dates = data[
        data['paid_at'] < data['joined_at']
    ]

    print(f'{name}:')
    print(f'Платежей до регистрации: {len(invalid_dates)}')
    print(f'Их сумма: {invalid_dates["revenue"].sum():.0f}')
    print()

In [ ]:
# Создаём копию транзакций Control 2
df = transactions_control_2.copy()

# Преобразуем даты
df['joined_at'] = pd.to_datetime(df['joined_at'])
df['paid_at'] = pd.to_datetime(df['paid_at'])

# Отбираем платежи до регистрации
early_payments = df[
    df['paid_at'] < df['joined_at']
].copy()

# Рассчитываем разницу в днях
early_payments['days_before'] = (
    early_payments['joined_at'] - early_payments['paid_at']
).dt.days

# Показываем 10 самых больших расхождений
display(
    early_payments[
        ['uid', 'joined_at', 'paid_at', 'days_before', 'revenue']
    ].sort_values('days_before', ascending=False).head(10)
)

**Результаты проверки**

Обнаружены транзакции с датой платежа раньше регистрации пользователя:

- Test — 47 транзакций.
- Control 1 — 55 транзакций.
- Control 2 — 60 транзакций.

Всего выявлено 162 подозрительные транзакции.

При анализе обнаружена закономерность: в ряде записей число `11` находится на месте месяца или дня, что позволяет предположить ошибку преобразования формата дат.

Например, значение `2017-02-11` могло возникнуть вместо `2017-11-02` при перестановке месяца и дня.

Для проверки гипотезы было предложено преобразование неоднозначных дат.


In [ ]:
# Функция для исправления неоднозначных дат
def fix_date(value):
    date_part, time_part = value.split(' ')
    year, month, day = date_part.split('-')

    if day == '11' and month != '11':
        month, day = day, month

    return f'{year}-{month}-{day} {time_part}'


# Создаём таблицы с исправленными датами
transactions_dates_fixed = {}

for name, df in transactions.items():
    data = df.copy()

    for col in ['joined_at', 'paid_at']:
        data[col] = pd.to_datetime(
            data[col].apply(fix_date)
        )

    transactions_dates_fixed[name] = data

    invalid = (data['paid_at'] < data['joined_at']).sum()

    print(f'{name}: {invalid} платежей до регистрации')

**Результаты проверки гипотезы**

После применения предложенного преобразования дат количество транзакций, предшествующих регистрации пользователей, сократилось до нуля во всех трёх группах.

Это согласуется с гипотезой об ошибке интерпретации формата дат.

Однако устранение хронологических противоречий само по себе не доказывает, что все даты восстановлены правильно. Предложенное правило требует дополнительной проверки, поскольку может затронуть и корректные записи.

**Вывод:** обнаруженные аномалии могут быть связаны с форматом дат, а не с наличием ошибочных платежей. Поэтому транзакции не исключались автоматически только на основании нарушения хронологии.

Исходные даты сохранены, а исправленная версия рассматривается как дополнительная гипотеза для анализа.


### 3.6. Анализ операций возврата платежей

При сверке выручки обнаружены транзакции с пометкой `refund` в поле `from_page`.

Такие операции могут быть связаны с возвратами денежных средств и влиять на расчёт чистой выручки.

Проверим количество подобных записей, их суммы и распределение между экспериментальными группами.

Поскольку суммы транзакций положительные, необходимо учитывать, что название источника платежа само по себе не подтверждает факт возврата.


In [ ]:
# Ищем операции, похожие на возвраты

for name, df in transactions.items():

    refund_rows = df[
        df['from_page'].str.contains('refun', case=False, na=False)]

    print(f'{name}: {len(refund_rows)} подозрительных операций')
    print(f'Их сумма: {refund_rows["revenue"].sum():.0f}')

    display(refund_rows)

**Результаты проверки**

Операции с подстрокой `refun` в `from_page` обнаружены только в Control 1:

- Количество операций — 3.
- Общая сумма — 806 единиц.
- Все операции принадлежат одному пользователю.

Значения поля `from_page`:
- `refund_VP` — 26.
- `refund_WP` — 143.
- `refung_AP` — 637.

Для данного пользователя значение `total_revenue` составляет 26 единиц, тогда как сумма транзакций равна 806.

Разница в 780 единиц совпадает с суммой двух операций: 143 и 637.

**Вывод:** обнаруженные операции могут объяснять одно из расхождений между источниками выручки. Однако без дополнительной информации нельзя однозначно определить их экономический смысл.

Поэтому данные операции не исключались и не вычитались автоматически из выручки. Неопределённость их интерпретации учитывается как ограничение исследования.


### 3.7. Итоги предобработки данных

В ходе проверки качества данных:

1. Подтверждена уникальность пользователей и отсутствие пересечений между экспериментальными группами.
2. Исключены 630 некорректных строк транзакций Control 1.
3. Исследованы полные дубликаты платежей и их влияние на выручку.
4. Обнаружены аномалии дат платежей и предложена гипотеза об ошибке формата дат.
5. Проведена сверка выручки между пользовательскими и транзакционными данными.
6. Исследованы операции с пометкой `refund`.

Неоднозначные записи сохранены для основного анализа там, где не получено достаточных оснований для их исключения.

При интерпретации результатов A/B-теста необходимо учитывать возможное влияние спорных транзакций, крупных покупок и ограничений исходных данных.


## 4. Проверка корректности A/B-теста

Перед сравнением продуктовых метрик необходимо убедиться, что экспериментальные группы сопоставимы по основным характеристикам пользователей.

Существенные различия в составе групп могут влиять на конверсию и выручку независимо от изменения стоимости подписки.

В рамках проверки рассмотрим:

- Размеры экспериментальных групп.
- Распределение пользователей по странам.
- Возраст и пол пользователей.
- Результаты A/A-проверки двух контрольных групп.

### 4.1. Сравнение характеристик пользователей

На первом этапе исследуем распределение пользователей по странам, возрасту и полу. Это позволит выявить возможные различия в составе экспериментальных групп.

In [ ]:
# Объединяем пользователей трёх групп
users_all = pd.concat(
    [df.assign(group=name) for name, df in users.items()],
    ignore_index=True)

# Рассчитываем долю каждой страны внутри группы
country_share = pd.crosstab(
    users_all['country'],
    users_all['group'],
    normalize='columns') * 100

# Выводим таблицу
display(country_share.round(2))

In [ ]:
# Находим 10 стран с наибольшим количеством пользователей
top_countries = users_all['country'].value_counts().head(10).index

# Отбираем их доли в каждой группе
top_country_share = country_share.loc[top_countries].copy()

# Объединяем остальные страны в категорию Other
top_country_share.loc['Other'] = (
    100 - top_country_share.sum()
)

# Выводим результат
display(top_country_share.round(2))

In [ ]:
# Сравниваем основные характеристики групп
for name, df in users.items():
    print(f'\n{name}')
    print('Пользователей:', len(df))
    print('Средний возраст:', round(df['age'].mean(), 2))
    print('Медианный возраст:', df['age'].median())
    print('Распределение пола:')
    print(df['gender'].value_counts(normalize=True).round(3))

In [ ]:
# Сравниваем распределение пола между группами
for name, df in users.items():
    print(f'\n{name}')
    print(
        (df['gender'].value_counts(normalize=True) * 100).round(2)
    )

### 4.2. A/A-проверка контрольных групп

Для дополнительной проверки корректности эксперимента сравним две контрольные группы — Control 1 и Control 2.

В обеих группах стоимость Premium-подписки оставалась неизменной, поэтому систематических различий в ключевых продуктовых метриках не ожидается.

Проверим следующие показатели:

- Конверсию в покупку Premium.
- Общий ARPU.
- ARPU Premium.

Для проверки конверсии используем критерий хи-квадрат, для сравнения денежных метрик — тест Уэлча.

Уровень статистической значимости — 0,05.

**Нулевая гипотеза (H₀):** значения проверяемой метрики в контрольных группах не различаются.

**Альтернативная гипотеза (H₁):** значения метрики различаются.

In [ ]:
# A/A: сравнение конверсии Premium в двух контрольных группах
premium_types = ['premium_no_trial', 'trial_premium']
buyers_1 = transactions_control_1_clean.loc[
    transactions_control_1_clean['product_type'].isin(premium_types), 'uid'
].nunique()
buyers_2 = transactions_control_2.loc[
    transactions_control_2['product_type'].isin(premium_types), 'uid'
].nunique()
n_1, n_2 = len(users_control_1), len(users_control_2)
table = [[buyers_1, n_1 - buyers_1], [buyers_2, n_2 - buyers_2]]
chi2, p_value, _, _ = chi2_contingency(table, correction=False)
print(f'A/A — конверсия Premium: p-value = {p_value:.4f}')

In [ ]:
# A/A-проверка денежных метрик: выполняется независимо от последующих A/B-проверок
_, p_total = ttest_ind(
    users_control_1['total_revenue'], users_control_2['total_revenue'],
    equal_var=False
)

def premium_per_user(df_users, df_transactions):
    premiums = df_transactions.loc[
        df_transactions['product_type'].isin(premium_types)
    ]
    revenue = premiums.groupby('uid')['revenue'].sum()
    return df_users['uid'].map(revenue).fillna(0)

control_1_premium = premium_per_user(users_control_1, transactions_control_1_clean)
control_2_premium = premium_per_user(users_control_2, transactions_control_2)
_, p_premium = ttest_ind(control_1_premium, control_2_premium, equal_var=False)
print(f'A/A — общий ARPU: p-value = {p_total:.4f}')
print(f'A/A — ARPU Premium: p-value = {p_premium:.4f}')

**Результаты A/A-проверки**

| Метрика | p-value | Статистически значимое различие |
|---|---:|---|
| Конверсия Premium | 0,6423 | Нет |
| Общий ARPU | 0,2628 | Нет |
| ARPU Premium | 0,5341 | Нет |

По всем трём метрикам p-value превышает 0,05. Следовательно, оснований отвергать нулевые гипотезы не получено.

**Вывод:** статистически значимых различий между контрольными группами по проверенным показателям не обнаружено.

При этом отсутствие значимости не доказывает полную эквивалентность групп. Результаты A/A-проверки рассматриваются как дополнительная диагностическая информация.

### 4.3. Итоги проверки корректности эксперимента

В ходе проверки экспериментальных групп получены следующие результаты:

1. **Размеры групп сопоставимы.** В Test участвуют 4 308 пользователей, в Control 1 — 4 340, в Control 2 — 4 264.

2. **Пересечений между группами не обнаружено.** Каждый пользователь относится только к одной экспериментальной группе.

3. **Характеристики пользователей визуально схожи.** Распределение по крупнейшим странам, средний возраст и соотношение полов не демонстрируют очевидных существенных перекосов.

4. **A/A-проверка не выявила статистически значимых различий** между Control 1 и Control 2 по конверсии в Premium, общему ARPU и ARPU Premium.

**Вывод**

Проведённые проверки не выявили явных признаков некорректного разделения пользователей на экспериментальные группы.

При этом отсутствие статистически значимых различий не доказывает полной эквивалентности групп или корректности рандомизации.

Дополнительным ограничением остаётся отсутствие подтверждённого перечня стран и платёжных систем, для которых действовала новая стоимость подписки. Поэтому основные результаты рассматриваются как сравнение экспериментальных групп в целом, а не как оценка эффекта исключительно среди пользователей, непосредственно столкнувшихся с новой ценой.

## 5. Анализ продуктовых метрик

Для оценки результатов A/B-теста рассчитаем ключевые показатели монетизации приложения.

**Основные метрики исследования:**

- **Conversion Rate** — доля пользователей, совершивших хотя бы одну покупку Premium.
- **ARPU** — средняя выручка на одного пользователя экспериментальной группы, включая пользователей без покупок.
- **ARPPU** — средняя выручка на одного платящего пользователя.

Дополнительно выделим конверсию в `premium_no_trial`, поскольку изменение цены затрагивало обычную Premium-подписку.

Метрики рассчитаем отдельно для Test, Control 1 и Control 2, чтобы оценить направление и величину наблюдаемых различий.

Для общего ARPU используем поле `total_revenue`, а для показателей Premium — соответствующие транзакции. Обнаруженные ранее расхождения между источниками учитываем как ограничение исследования.

### Воспроизводимость и правила использования данных

Исходные CSV-файлы не изменяются. До удаления пропусков исследуются исходные таблицы транзакций. После диагностики для последующих расчётов используется `transactions`, содержащий очищенную от 630 некорректных записей таблицу Control 1; полные повторения платежей **не удаляются**, поскольку их происхождение не установлено. Выручка `total_revenue` из таблиц пользователей используется для общего ARPU, транзакции — для выручки Premium. Исправленные даты сохраняются отдельно и не подменяют даты в основной таблице.

**Граница интерпретации:** неизвестны точные страны и платёжные системы, на которые распространялась новая цена. Поэтому сравнение проводится по всем пользователям групп (анализ по назначенной группе, при условии корректного исходного разделения), а не подтверждённой оценкой эффекта среди фактически увидевших новую цену. Результаты по отдельным платёжным системам и странам носят исследовательский характер. Без информации о рандомизации и полном окне наблюдения причинную интерпретацию следует делать осторожно.

In [ ]:
# Анализ выручки по типам продуктов

for name, df in transactions.items():

    result = df.groupby('product_type')['revenue'].agg(
        ['count', 'sum'])

    print(f'\n{name}')
    display(result)

In [ ]:
# Выбираем типы премиум-подписок
premium_types = ['premium_no_trial', 'trial_premium']

for name, df in transactions.items():

    # Оставляем только премиум-платежи
    premium = df[df['product_type'].isin(premium_types)]

    # Количество пользователей группы
    total_users = len(users[name])

    # Уникальные покупатели Premium
    buyers = premium['uid'].nunique()

    # Конверсия в покупку Premium
    conversion = buyers / total_users * 100

    # Выручка от Premium на пользователя группы
    premium_arpu = premium['revenue'].sum() / total_users

    print(f'{name}:')
    print(f'Покупателей Premium: {buyers}')
    print(f'Конверсия: {conversion:.2f}%')
    print(f'ARPU Premium: {premium_arpu:.2f}')
    print()

In [ ]:
# Рассчитываем ARPPU Premium для каждой группы

premium_types = ['premium_no_trial', 'trial_premium']

for name, df in transactions.items():

    # Выбираем премиум-транзакции
    premium = df[df['product_type'].isin(premium_types)]

    # Считаем уникальных покупателей
    buyers = premium['uid'].nunique()

    # Считаем выручку Premium
    revenue = premium['revenue'].sum()

    # Средняя выручка на платящего пользователя
    arppu = revenue / buyers if buyers > 0 else 0

    print(f'{name}: ARPPU Premium = {arppu:.2f}')

In [ ]:
# Сравниваем метрики по типам подписок
results = []

for name, df in transactions.items():
    for product in ['premium_no_trial', 'trial_premium']:

        data = df[df['product_type'] == product]

        buyers = data['uid'].nunique()
        revenue = data['revenue'].sum()
        total_users = len(users[name])

        results.append({
            'group': name,
            'product': product,
            'buyers': buyers,
            'conversion': buyers / total_users * 100,
            'ARPU': revenue / total_users,
            'ARPPU': revenue / buyers if buyers else 0
        })

premium_metrics = pd.DataFrame(results)

display(premium_metrics.round(2))

In [ ]:
# Итоговая таблица продуктовых метрик

summary = []

for name in ['Test', 'Control 1', 'Control 2']:

    df_users = users[name]
    df_tx = transactions[name]

    # Покупки Premium
    premium = df_tx[
        df_tx['product_type'].isin(premium_types)
    ]

    # Premium без пробного периода
    no_trial = df_tx[
        df_tx['product_type'] == 'premium_no_trial'
    ]

    total_users = len(df_users)
    premium_buyers = premium['uid'].nunique()

    summary.append({
        'Группа': name,
        'Пользователи': total_users,
        'Конверсия Premium, %':
            premium_buyers / total_users * 100,
        'Конверсия без trial, %':
            no_trial['uid'].nunique() / total_users * 100,
        'Общий ARPU':
            df_users['total_revenue'].mean(),
        'ARPU Premium':
            premium['revenue'].sum() / total_users,
        'ARPPU Premium':
            premium['revenue'].sum() / premium_buyers
            if premium_buyers else 0
    })

summary_df = pd.DataFrame(summary)

display(summary_df.round(2))

### 5.1. Результаты сравнения продуктовых метрик

По результатам анализа получены следующие наблюдения.

**1. Конверсия в Premium**

Конверсия в покупку Premium в тестовой группе составила 2,95%, тогда как в Control 1 — 3,82%, а в Control 2 — 3,64%.

Таким образом, в Test наблюдается более низкая доля пользователей, совершивших покупку Premium.

Для подписки без пробного периода различия ещё заметнее: 1,56% в Test против 2,28% и 2,39% в контрольных группах.

**2. ARPU Premium**

Средняя выручка от Premium на пользователя составила:

- Test — 373,93.
- Control 1 — 357,60.
- Control 2 — 325,70.

Несмотря на более низкую конверсию, в тестовой группе наблюдается более высокий ARPU Premium.

**3. ARPPU Premium**

Средняя выручка на платящего пользователя Premium:

- Test — 12 684,31.
- Control 1 — 9 349,27.
- Control 2 — 8 959,94.

Это означает, что покупатели Premium в Test в среднем приносили больше выручки, чем покупатели контрольных групп.

**4. Общий ARPU**

Общий ARPU составил 534,08 в Test, 594,76 в Control 1 и 450,38 в Control 2.

Таким образом, тестовая группа превосходит Control 2, но уступает Control 1 по общей выручке на пользователя.

### Промежуточный вывод

В тестовой группе наблюдается снижение конверсии в покупку Premium при одновременном увеличении средней выручки от платящих пользователей.

Такая картина согласуется с возможным влиянием повышения цены: покупок становится меньше, а выручка с одного платящего пользователя — выше. Однако без статистических проверок нельзя установить, является ли рост средней выручки устойчивым.

**Вывод:** сравнение продуктовых метрик само по себе недостаточно для решения о внедрении новой цены. Результаты статистических тестов приведены в следующем разделе.

## 6. Статистический анализ результатов A/B-теста

### 6.1. Проверка статистических гипотез

На предыдущем этапе были обнаружены различия в конверсии и выручке между тестовой и контрольными группами.

Для оценки статистической значимости этих различий проведём проверку гипотез.

**Основные гипотезы:**

- **H₀ (нулевая гипотеза):** значения исследуемой метрики в тестовой и контрольной группах одинаковы.
- **H₁ (альтернативная гипотеза):** значения исследуемой метрики различаются.

Для проверки конверсии используем критерий хи-квадрат, а для сравнения средней выручки на пользователя — тест Уэлча.

Дополнительно применим бутстрэп для построения доверительных интервалов разницы ARPU.

Номинальный уровень статистической значимости составляет 0,05. Поскольку проводятся несколько сравнений, при интерпретации результатов учитываем риск множественного тестирования.

Особое внимание уделим влиянию крупных покупателей, поскольку распределение выручки характеризуется высокой долей нулевых значений и наличием выбросов.

In [ ]:
# A/B-тест: сравниваем конверсию в Premium

premium_types = ['premium_no_trial', 'trial_premium']

# Рассчитываем уникальных покупателей в каждой группе
buyers = {}

for name, df in transactions.items():
    premium = df[df['product_type'].isin(premium_types)]
    buyers[name] = premium['uid'].nunique()

# Сравниваем Test с каждой контрольной группой
for control in ['Control 1', 'Control 2']:

    table = [
        [buyers['Test'], len(users['Test']) - buyers['Test']],
        [buyers[control], len(users[control]) - buyers[control]]
    ]

    chi2, p_value, _, _ = chi2_contingency(
        table, correction=False
    )

    print(f'Test vs {control}: p-value = {p_value:.4f}')

In [ ]:
# Проверяем конверсию в Premium без пробного периода
buyers_no_trial = {}

for name, df in transactions.items():
    premium = df[
        df['product_type'] == 'premium_no_trial'
    ]

    buyers_no_trial[name] = premium['uid'].nunique()

# Сравниваем Test с контрольными группами
for control in ['Control 1', 'Control 2']:

    table = [
        [
            buyers_no_trial['Test'],
            len(users['Test']) - buyers_no_trial['Test']
        ],
        [
            buyers_no_trial[control],
            len(users[control]) - buyers_no_trial[control]
        ]
    ]

    chi2, p_value, _, _ = chi2_contingency(
        table, correction=False
    )

    print(f'Test vs {control}: p-value = {p_value:.4f}')

In [ ]:
# Доверительные интервалы разницы конверсий

for control in ['Control 1', 'Control 2']:

    ci_low, ci_high = confint_proportions_2indep(
        buyers_no_trial['Test'],
        len(users['Test']),
        buyers_no_trial[control],
        len(users[control]),
        compare='diff',
        method='newcomb',
        alpha=0.025
    )

    print(f'Test vs {control}')
    print(f'97.5% ДИ: [{ci_low*100:.2f}; {ci_high*100:.2f}] п.п.')
    print()

In [ ]:
# Рассчитываем общий ARPU для трёх групп

for name, df in users.items():

    total_users = len(df)
    total_revenue = df['total_revenue'].sum()

    arpu = total_revenue / total_users

    print(f'{name}:')
    print(f'Пользователей: {total_users}')
    print(f'Общая выручка: {total_revenue}')
    print(f'ARPU: {arpu:.2f}')
    print()

In [ ]:
# Изучаем распределение выручки по пользователям

for name, df in users.items():
    print(f'\n{name}')

    print(
        df['total_revenue'].describe(
            percentiles=[0.5, 0.9, 0.95, 0.99]
        )
    )

    print(
        'Пользователей без выручки:',
        (df['total_revenue'] == 0).sum())

In [ ]:
# Проверяем влияние крупнейшего покупателя на ARPU

for name, df in users.items():

    revenue = df['total_revenue']

    # Исходный ARPU
    arpu = revenue.mean()

    # Максимальная выручка одного пользователя
    max_revenue = revenue.max()

    # ARPU без одного крупнейшего покупателя
    arpu_without_top = (
        (revenue.sum() - max_revenue) / (len(revenue) - 1)
    )

    print(f'\n{name}')
    print(f'Исходный ARPU: {arpu:.2f}')
    print(f'Максимальный платёж пользователя: {max_revenue:.0f}')
    print(f'ARPU без крупнейшего покупателя: {arpu_without_top:.2f}')

In [ ]:
# Выручка каждого пользователя тестовой группы
test_revenue = users_test['total_revenue']

# Сравниваем Test с двумя контрольными группами
for name in ['Control 1', 'Control 2']:

    control_revenue = users[name]['total_revenue']

    # Тест Уэлча
    t_stat, p_value = ttest_ind(
        test_revenue,
        control_revenue,
        equal_var=False
    )

    print(f'Test vs {name}')
    print(f'p-value: {p_value:.4f}')
    print()

In [ ]:
# Генератор случайных чисел
rng = np.random.default_rng(42)

test = users_test['total_revenue'].to_numpy()

for name in ['Control 1', 'Control 2']:
    control = users[name]['total_revenue'].to_numpy()

    differences = []

    # 3000 бутстрэп-итераций
    for i in range(3000):
        test_sample = rng.choice(test, size=len(test), replace=True)
        control_sample = rng.choice(control, size=len(control), replace=True)

        diff = test_sample.mean() - control_sample.mean()
        differences.append(diff)

    ci_low, ci_high = np.percentile(differences, [2.5, 97.5])

    print(f'\nTest vs {name}')
    print(f'Разница ARPU: {test.mean() - control.mean():.2f}')
    print(f'95% ДИ: [{ci_low:.2f}, {ci_high:.2f}]')

In [ ]:
# Создаём массив выручки Premium для каждого пользователя
premium_revenue = {}

for name, df in transactions.items():

    # Оставляем только премиум-платежи
    premium = df[df['product_type'].isin(premium_types)]

    # Суммируем выручку по пользователям
    payments = premium.groupby('uid')['revenue'].sum()

    # Добавляем нулевую выручку тем, кто не покупал Premium
    premium_revenue[name] = (
        users[name]['uid'].map(payments).fillna(0)
    )

# Проверяем статистическую значимость ARPU Premium
for control in ['Control 1', 'Control 2']:

    t_stat, p_value = ttest_ind(
        premium_revenue['Test'],
        premium_revenue[control],
        equal_var=False
    )

    print(f'Test vs {control}: p-value = {p_value:.4f}')

In [ ]:
# Бутстрэп для ARPU Premium
rng = np.random.default_rng(42)

test = premium_revenue['Test'].to_numpy()

for name in ['Control 1', 'Control 2']:

    control = premium_revenue[name].to_numpy()
    differences = []

    for i in range(3000):

        test_sample = rng.choice(
            test, size=len(test), replace=True
        )

        control_sample = rng.choice(
            control, size=len(control), replace=True
        )

        diff = test_sample.mean() - control_sample.mean()
        differences.append(diff)

    ci_low, ci_high = np.percentile(
        differences, [2.5, 97.5]
    )

    print(f'\nTest vs {name}')
    print(f'Разница ARPU Premium: {test.mean() - control.mean():.2f}')
    print(f'95% ДИ: [{ci_low:.2f}, {ci_high:.2f}]')

In [ ]:
# Анализируем платёжные системы для Premium без пробного периода

for name, df in transactions.items():

    premium = df[
        df['product_type'] == 'premium_no_trial'
    ]

    result = premium.groupby('payment_id').agg(
        transactions=('uid', 'count'),
        buyers=('uid', 'nunique'),
        revenue=('revenue', 'sum'),
        avg_payment=('revenue', 'mean')
    )

    print(f'\n{name}')
    display(result.round(2))

In [ ]:
# Объединяем транзакции Premium без пробного периода
premium_payments = pd.concat(
    [
        df[df['product_type'] == 'premium_no_trial']
        .assign(group=name)
        for name, df in transactions.items()
    ],
    ignore_index=True
)

# Оставляем платежную систему 147
payments_147 = premium_payments[
    premium_payments['payment_id'] == 147
]

# Выбираем 8 стран с наибольшим числом платежей
top_countries = payments_147['country'].value_counts().head(8).index

# Сравниваем средние платежи по странам и группам
country_payments = (
    payments_147[
        payments_147['country'].isin(top_countries)
    ]
    .groupby(['country', 'group'])['revenue']
    .agg(['count', 'mean'])
    .round(2)
)

display(country_payments)

In [ ]:
# Изучаем суммы покупок Premium через систему 147

for country in ['Argentina', 'Italy', 'Spain']:

    print(f'\n{country}')

    country_data = payments_147[
        payments_147['country'] == country
    ]

    result = pd.crosstab(
        country_data['revenue'],
        country_data['group']
    )

    display(result)

In [ ]:
# Собираем покупателей Premium без пробного периода
premium_buyers = pd.concat([
    df[df['product_type'] == 'premium_no_trial']
    [['uid']].drop_duplicates().assign(group=name)
    for name, df in transactions.items()
], ignore_index=True)

premium_buyers['bought_premium'] = 1

# Добавляем информацию о покупке ко всем пользователям
country_data = users_all.merge(
    premium_buyers,
    on=['uid', 'group'],
    how='left'
)

# Те, кто не покупал Premium, получают 0
country_data['bought_premium'] = (
    country_data['bought_premium'].fillna(0)
)

# Выбираем 10 стран с наибольшим числом пользователей
top_10 = country_data['country'].value_counts().head(10).index

# Рассчитываем конверсию по странам и группам
country_conversion = (
    country_data[country_data['country'].isin(top_10)]
    .groupby(['country', 'group'])
    .agg(
        users=('uid', 'count'),
        buyers=('bought_premium', 'sum'),
        conversion=('bought_premium', 'mean')
    )
    .reset_index()
)

country_conversion['conversion'] *= 100

display(country_conversion.round(2))

### 6.2. Статистическая проверка конверсии

Для оценки изменений в покупательском поведении сравним конверсию в Premium между Test и контрольными группами.

**Результаты проверки общей конверсии в Premium**

| Сравнение | p-value |
|---|---:|
| Test vs Control 1 | 0,0242 |
| Test vs Control 2 | 0,0746 |

При использовании поправки Бонферрони для двух сравнений порог значимости составляет 0,025.

В первом сравнении различие проходит этот порог, во втором — нет.

**Результаты проверки конверсии в Premium без пробного периода**

| Сравнение | p-value | 97,5% доверительный интервал разницы |
|---|---:|---|
| Test vs Control 1 | 0,0139 | [−1,40; −0,06] п.п. |
| Test vs Control 2 | 0,0053 | [−1,53; −0,16] п.п. |

В обоих сравнениях наблюдается статистически значимое снижение конверсии в Premium без пробного периода при поправке на два сравнения.

**Вывод:** тестовая группа демонстрирует более низкую конверсию в покупку Premium без пробного периода относительно обеих контрольных групп.

Однако эти проверки были частью более широкого исследовательского анализа. Поэтому результаты следует рассматривать с учётом множественных проверок и ограничений определения целевой аудитории эксперимента.

### 6.3. Статистическая проверка ARPU

Для сравнения средней выручки на пользователя использован тест Уэлча, а для оценки неопределённости разницы средних — процентильный бутстрэп (3 000 итераций, 95% интервалы).

**Общий ARPU**

| Сравнение | Разница Test − Control | p-value | 95% ДИ |
|---|---:|---:|---|
| Test vs Control 1 | −60,68 | 0,6609 | [−371,21; +182,20] |
| Test vs Control 2 | +83,70 | 0,2937 | [−69,30; +246,25] |

**ARPU Premium**

| Сравнение | Разница Test − Control | p-value | 95% ДИ |
|---|---:|---:|---|
| Test vs Control 1 | +16,34 | 0,7888 | [−98,90; +138,66] |
| Test vs Control 2 | +48,23 | 0,4182 | [−65,18; +168,71] |

Оба доверительных интервала по каждой денежной метрике включают ноль; статистически значимого увеличения ARPU не обнаружено. Отсутствие значимости не доказывает отсутствие эффекта. Распределения сильно асимметричны, поэтому приведён анализ чувствительности. Указанные 95% интервалы не скорректированы на все выполненные исследовательские сравнения.

In [ ]:
# Анализ чувствительности ARPU Premium к полным повторам транзакций
# Это альтернативный сценарий, а НЕ изменение основного датасета.
rows = []
for name in ['Test', 'Control 1', 'Control 2']:
    source = transactions[name]
    no_exact_duplicates = source.drop_duplicates()
    n_users = len(users[name])
    scenarios = [
        ('Повторы сохранены', source),
        ('Полные повторы удалены', no_exact_duplicates),
    ]
    for scenario, data in scenarios:
        premium = data[data['product_type'].isin(premium_types)]
        buyers = premium['uid'].nunique()
        rows.append({
            'Группа': name,
            'Сценарий': scenario,
            'Покупатели Premium': buyers,
            'Конверсия Premium, %': buyers / n_users * 100,
            'ARPU Premium': premium['revenue'].sum() / n_users,
        })
sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df.round(2))

**Проверка чувствительности к полным дубликатам.** Дополнительно рассчитаны конверсия и ARPU Premium для двух сценариев — с сохранёнными и удалёнными полными повторами транзакций. Это диагностическая проверка, а не основание удалять покупки: одинаковые строки могут описывать реальные повторные платежи. Сопоставление показывает, насколько выбор правила очистки влияет на оценку денежной метрики. Для окончательного решения нужны уникальные идентификаторы транзакций и описание бизнес-логики платежей.

### 6.4. Анализ влияния крупных покупателей

При исследовании распределения выручки обнаружена высокая доля пользователей без покупок.

Медиана и 95-й перцентиль `total_revenue` равны нулю во всех экспериментальных группах.

При этом отдельные пользователи совершают покупки на значительно более крупные суммы.

Наиболее заметный пример — пользователь Control 1 с общей выручкой 486 330 единиц, что составляет около 18,8% выручки данной группы.

Для оценки влияния крупных покупателей проведён анализ чувствительности: ARPU пересчитан после исключения одного пользователя с максимальной выручкой из каждой группы.

| Группа | Исходный ARPU | ARPU без крупнейшего покупателя |
|---|---:|---:|
| Test | 534,08 | 504,05 |
| Control 1 | 594,76 | 482,82 |
| Control 2 | 450,38 | 431,30 |

После исключения крупнейших покупателей соотношение Test и Control 1 изменилось: исходно ARPU Test был ниже, а после исключения — выше.

Это свидетельствует о высокой чувствительности среднего значения к отдельным крупным покупкам.

**Вывод:** отдельные крупные платежи существенно влияют на результаты сравнения общей выручки.

При этом такие пользователи не исключаются из основного анализа без подтверждённых причин считать их операции ошибочными. Пересчёт используется исключительно как дополнительная проверка устойчивости результатов.

### 6.5. Итоги статистического анализа

Проведённые статистические проверки показали:

1. Конверсия в Premium без пробного периода в тестовой группе ниже обеих контрольных групп. Различия статистически значимы в проведённых сравнениях с поправкой Бонферрони на два контроля.

2. ARPU Premium в тестовой группе выше, однако статистически значимого роста выручки не обнаружено.

3. Общий ARPU не продемонстрировал статистически значимых различий между Test и контрольными группами.

4. Денежные метрики чувствительны к крупным покупкам, что необходимо учитывать при интерпретации результатов.

5. A/A-проверка двух контрольных групп не выявила статистически значимых различий по рассмотренным метрикам.

**Общий вывод**

На основании проведённого анализа нельзя подтвердить положительное влияние изменения стоимости Premium-подписки на выручку приложения.

При этом наблюдается снижение конверсии в покупку Premium без пробного периода.

Полученные результаты не дают достаточных оснований рекомендовать массовое внедрение новой цены.

Данный вывод предварительный, поскольку исследование включает дополнительные проверки, проведённые после просмотра данных, а также ограничения, связанные с неоднозначными транзакциями и определением целевой аудитории эксперимента.

## 7. Визуализация результатов

Для наглядного представления результатов A/B-теста построим графики ключевых продуктовых метрик.

Визуализации помогут сравнить экспериментальные группы, оценить направление наблюдаемых различий и показать статистическую неопределённость полученных результатов.

### 7.1. Конверсия в Premium без пробного периода

Сравним долю пользователей, совершивших покупку `premium_no_trial`, в тестовой и контрольных группах.

In [ ]:
groups = summary_df['Группа']
conversion = summary_df['Конверсия без trial, %']

plt.figure(figsize=(9, 5))

bars = plt.bar(groups, conversion)

plt.title('Конверсия в Premium без пробного периода')
plt.ylabel('Конверсия, %')
plt.ylim(0, 3)

for bar, value in zip(bars, conversion):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f'{value:.2f}%',
        ha='center'
    )

plt.tight_layout()
plt.show()

**Интерпретация графика**

Конверсия в покупку Premium без пробного периода в тестовой группе составляет 1,56%, что ниже показателей обеих контрольных групп — 2,28% и 2,39%.

Наблюдаемое снижение конверсии подтверждается проведёнными статистическими проверками с поправкой на два контрольных сравнения.

При этом статистическая значимость различий сама по себе не доказывает, что причиной снижения стало именно изменение цены подписки.

In [ ]:
groups = summary_df['Группа']
x = np.arange(len(groups))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(
    x - width / 2,
    summary_df['Общий ARPU'],
    width,
    label='Общий ARPU'
)

bars2 = ax.bar(
    x + width / 2,
    summary_df['ARPU Premium'],
    width,
    label='ARPU Premium'
)

ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel('Выручка на пользователя')
ax.set_title('Сравнение ARPU по экспериментальным группам')
ax.legend()


# Добавляем числовые подписи над столбцами
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 5,
            f'{height:.1f}',
            ha='center')

plt.tight_layout()
plt.show()

### 7.2. Сравнение общего ARPU и ARPU Premium

Общий ARPU тестовой группы составляет 534,08, что ниже Control 1 (594,76), но выше Control 2 (450,38).

При этом ARPU Premium тестовой группы составляет 373,93 и превышает значения обеих контрольных групп.

Таким образом, направление различий зависит от рассматриваемой денежной метрики.

**Вывод:** рост ARPU Premium не сопровождается однозначным увеличением общей выручки на пользователя. Статистические проверки не выявили значимого роста ни одной из этих денежных метрик.

### 7.3. Доверительные интервалы изменения ARPU Premium

Для оценки неопределённости результатов визуализируем разницу ARPU Premium между тестовой и контрольными группами.

Доверительные интервалы рассчитаны методом бутстрэпа на основе 3 000 повторных выборок.

На графике нулевое значение означает отсутствие разницы между группами. Если доверительный интервал включает ноль, данные не позволяют уверенно установить направление эффекта.

In [ ]:
# Строим интервалы из рассчитанных пользовательских данных, а не из вручную заданных чисел
rng = np.random.default_rng(42)
controls = ['Control 1', 'Control 2']
diffs, ci_low, ci_high = [], [], []
test_values = premium_revenue['Test'].to_numpy()
for control_name in controls:
    control_values = premium_revenue[control_name].to_numpy()
    values = []
    for _ in range(3000):
        a = rng.choice(test_values, len(test_values), replace=True)
        b = rng.choice(control_values, len(control_values), replace=True)
        values.append(a.mean() - b.mean())
    diff = test_values.mean() - control_values.mean()
    low, high = np.percentile(values, [2.5, 97.5])
    diffs.append(diff)
    ci_low.append(low)
    ci_high.append(high)
errors = [[d - low for d, low in zip(diffs, ci_low)],
          [high - d for d, high in zip(diffs, ci_high)]]
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(diffs, controls, xerr=errors, fmt='o', capsize=6)
ax.axvline(0, linestyle='--', color='gray')
ax.set_title('Разница ARPU Premium: Test − Control')
ax.set_xlabel('Разница ARPU Premium')
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

**Интерпретация результатов**

Наблюдаемая разница ARPU Premium положительна в обоих сравнениях:

- Test vs Control 1: +16,34.
- Test vs Control 2: +48,23.

Однако оба 95% доверительных интервала пересекают нулевое значение.

Следовательно, данные совместимы как с отрицательным, так и с положительным эффектом изменения цены на ARPU Premium.

Статистически убедительного увеличения средней выручки от Premium на пользователя не обнаружено.

### 7.4. Основные результаты визуального анализа

Построенные графики демонстрируют следующие закономерности:

1. Конверсия в Premium без пробного периода в тестовой группе ниже обеих контрольных групп.
2. ARPU Premium в Test выше, однако общий ARPU не показывает однозначного преимущества тестовой группы.
3. Доверительные интервалы разницы ARPU Premium включают ноль, поэтому статистически подтверждённого роста выручки не обнаружено.

В совокупности результаты не дают достаточных оснований для вывода об экономической эффективности новой цены подписки.

## 8. Бизнес-выводы и рекомендации

### 8.1. Основные результаты эксперимента

В ходе исследования проанализированы данные 12 912 пользователей трёх экспериментальных групп.

**Конверсия в покупку Premium**

В тестовой группе конверсия в Premium без пробного периода составила 1,56%, тогда как в Control 1 и Control 2 — 2,28% и 2,39% соответственно.

Снижение конверсии статистически значимо в проведённых сравнениях с поправкой на два контроля.

**Выручка от Premium**

Несмотря на снижение конверсии, ARPU Premium в тестовой группе оказался выше:

- Test — 373,93.
- Control 1 — 357,60.
- Control 2 — 325,70.

Однако статистические проверки не подтвердили значимого увеличения ARPU Premium.

**Общая выручка**

Общий ARPU тестовой группы составил 534,08, что ниже Control 1 и выше Control 2.

Статистически значимых различий по общему ARPU также не обнаружено.

### 8.2. Бизнес-рекомендация

**На основании текущих результатов не рекомендуется массовое внедрение новой стоимости Premium-подписки.**

Повышение цены сопровождается снижением наблюдаемой конверсии в Premium без пробного периода, тогда как увеличение выручки на пользователя статистически не подтверждено.

При этом результаты не доказывают, что новая цена обязательно снижает выручку. Имеющихся данных недостаточно для уверенной оценки денежного эффекта.

Для окончательного решения рекомендуется подтвердить целевую аудиторию изменения цены и провести дополнительную проверку результатов.

### 8.3. Ограничения исследования

**1. Неопределённость охвата эксперимента**

Не подтверждён точный перечень стран и платёжных систем, на которые распространялась новая стоимость подписки.

Основной анализ проведён по экспериментальным группам целиком.

**2. Особенности качества данных**

В исходных таблицах выявлены полные повторы транзакций, неоднозначные даты платежей и расхождения между источниками выручки.

Для основного анализа повторяющиеся платежи сохранены. Их экономическая природа не подтверждена документально.

**3. Влияние крупных покупателей**

Выручка распределена неравномерно. Отдельные крупные покупки существенно влияют на средний ARPU, особенно в Control 1.

**4. Множественные статистические проверки**

В процессе исследования проведено несколько сравнений метрик и пользовательских сегментов.

Поправка Бонферрони применялась к отдельным парам сравнений, но не ко всему набору исследовательских гипотез. Поэтому результаты дополнительных проверок необходимо интерпретировать осторожно.

**5. Ограничения периода наблюдения**

Анализ не позволяет уверенно оценить долгосрочное влияние изменения цены на продления подписок, удержание и пожизненную ценность пользователей (LTV).

### 8.4. Рекомендации для дальнейшей работы

1. Уточнить перечень стран, платёжных систем и пользователей, которым была доступна новая цена.
2. Проверить источник и корректность спорных транзакций, включая дубликаты и неоднозначные даты.
3. Повторно оценить ARPU и конверсию для подтверждённой целевой аудитории эксперимента.
4. При необходимости провести дополнительный эксперимент с заранее определёнными основными метриками, уровнем значимости и необходимым объёмом выборки.
5. При наличии данных о более длительном периоде наблюдения оценить повторные платежи, удержание и LTV.

### 8.5. Итоговый вывод

Эксперимент показал снижение конверсии в покупку Premium без пробного периода в тестовой группе.

Несмотря на более высокий ARPU Premium, статистически убедительного увеличения выручки не обнаружено.

**Текущих результатов недостаточно для обоснования массового внедрения новой цены.**

Рекомендуется сохранить действующую стоимость подписки до дополнительной проверки данных и получения более надёжной оценки экономического эффекта.